This file will be for decideing which lr_scheduler to use

From previous files, the lr for head is 0.0017378008365631102 and lr range for whole model is ((1.9054607491852948e-06)/10, (1.9054607491852948e-06)/4)
The no. of epochs for head remains const. at 3 and no. of epochs for whole model is 8 as found in prev. file
The batch size found is 16 from prev. file, 
The weight decay value chosen is 1e-3
The optimizer chosen is RMSProp

I will be trying different lr schedulers 
they are StepLR, MultiStepLR, CosineAnnealingLR, CosineAnnealingWarmRestarts, OneCycleLR, SequentialLR, ReduceLROnPlateau

In [1]:
import pandas as pd
import regex as re
from fastai.vision.all import *
import torch

In [2]:
df = pd.read_csv(r'D:\Traffic\labels_processed.csv')

In [3]:
def label_function(dpath):
    class_name = re.findall(r'(\d+)_.*\.png$', dpath.name)
    class_id = int(class_name[0])
    return df['Names'][class_id]

In [5]:
path = Path(r'D:\Traffic\traffic_Data_processed\DATA')

In [6]:
lr_head = 0.0017378008365631102
lr_whole_model = 1.9054607491852948e-06

In [7]:
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
from torch.utils.data import random_split

train_size = int(0.75*len())

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=16,
    shuffle=False
)

In [ ]:
model = torchvision.models.resnet34(weights="DEFAULT")
model.fc = nn.Linear(
    model.fc.in_features,
    num_classes
)

In [ ]:
for param in model.parameters():
    param.requires_grad = False

for param in model.fc.parameters():
    param.requires_grad = True

In [ ]:
device = torch.device("cuda")

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.RMSprop(
    model.fc.parameters(),
    lr=lr_head,
    weight_decay=lr_head
)

scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=lr_head,
    epochs=3,
    steps_per_epoch=len(train_loader)
)

for epoch in range(3):
    model.train()
    train_loss = 0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        scheduler.step()          

        train_loss += loss.item()

        _, predicted = outputs.max(1)

        train_correct += (predicted == labels).sum().item()

        train_total += labels.size(0)

    train_acc = 100 * train_correct / train_total


    model.eval()

    valid_loss = 0
    valid_correct = 0
    valid_total = 0

    with torch.no_grad():

        for images, labels in valid_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            valid_loss += loss.item()

            _, predicted = outputs.max(1)

            valid_correct += (predicted == labels).sum().item()

            valid_total += labels.size(0)

    valid_acc = 100 * valid_correct / valid_total

    print(
        f"Epoch {epoch+1}/3 | "
        f"Train Acc: {train_acc:.2f}% | "
        f"Valid Acc: {valid_acc:.2f}%"
    )

In [ ]:
optimizer = torch.optim.RMSprop(
    model.fc.parameters(),
    lr=lr_head,
    weight_decay=1e-3
)

In [ ]:
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=lr_head,
    epochs=3,
    steps_per_epoch=len(train_loader)
)

In [ ]:
for epoch in range(3):

    train(...)

    validate(...)

    scheduler.step()

In [ ]:
for param in model.parameters():
    param.requires_grad = True

In [ ]:
optimizer = torch.optim.RMSprop(
    model.parameters(),
    lr=lr_whole_model,
    weight_decay=1e-3
)

In [ ]:
optimizer = RMSprop(

    [
        {"params": layer1.parameters(), "lr": ...},

        {"params": layer2.parameters(), "lr": ...},

        ...

        {"params": model.fc.parameters(), "lr": ...}
    ]
)

In [ ]:
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=[lr1, lr2, lr3, lr4, lr5],
    epochs=8,
    steps_per_epoch=len(train_loader)
)